In [ ]:
import time
import math
import warnings
import google.generativeai as genai

warnings.filterwarnings("ignore")

API_KEY = ""
genai.configure(api_key=API_KEY)

gen_model = genai.GenerativeModel('gemini-3.5-flash-lite')

embed_models = [
    m.name for m in genai.list_models()
    if 'embedContent' in m.supported_generation_methods
]
embed_model_name = 'models/text-embedding-004' if 'models/text-embedding-004' in embed_models else embed_models[0]

print(f"Generation model: gemini-3.5-flash-lite")
print(f"Embedding model: {embed_model_name}")

d:\pythone\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Адмін\AppData\Local\Temp\ipykernel_25376\161581644.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Generation model: gemini-3.5-flash-lite
Embedding model: models/gemini-embedding-001


In [2]:
documents = {
    "doc1_neural_networks.txt": [
        "An artificial neural network (ANN) is a computational model inspired by the structure of biological neural networks in the human brain. It consists of interconnected nodes called neurons organized into layers.",
        "The basic building block of a neural network is the perceptron. A perceptron takes multiple numerical inputs, multiplies each by a weight, adds a bias, and passes the result through an activation function.",
        "Common activation functions include ReLU (Rectified Linear Unit), Sigmoid, and Tanh. Non-linear activation functions allow neural networks to learn complex patterns in data."
    ],
    "doc2_training.txt": [
        "Training a neural network involves adjusting its weights and biases to minimize a loss function, which measures the difference between predicted and actual outputs.",
        "Backpropagation is the core algorithm used to compute the gradient of the loss function with respect to each weight using the chain rule of calculus.",
        "Gradient descent and its variants like Adam update the model weights in the opposite direction of the gradient using a hyperparameter called the learning rate."
    ],
    "doc3_architectures.txt": [
        "Convolutional Neural Networks (CNNs) are specialized for processing grid-like data such as images. They use convolutional filters to automatically detect spatial hierarchies of features.",
        "Recurrent Neural Networks (RNNs) are designed for sequential data like time series and text. They maintain a hidden state that captures information from previous steps.",
        "Long Short-Term Memory (LSTM) networks are an advanced type of RNN that solve the vanishing gradient problem using memory cells and gating mechanisms."
    ],
    "doc4_transformers.txt": [
        "The Transformer architecture relies entirely on the self-attention mechanism, discarding recurrence and convolutions to process sequence elements in parallel.",
        "Self-attention calculates attention scores between all pairs of tokens in an input sequence, enabling the model to capture long-range dependencies effectively.",
        "In our internal benchmark setup, the custom mini-transformer model NeuraLite-4 achieved 94.2% accuracy using 4 attention heads and a context window of 512 tokens."
    ]
}

chunks = []
chunk_counter = 1

for doc_name, doc_chunks in documents.items():
    print(f"Document: {doc_name}")
    for text in doc_chunks:
        chunk_id = f"chunk_{chunk_counter:02d}"
        chunks.append({
            "id": chunk_id,
            "source": doc_name,
            "text": text
        })
        print(f"  ├── {chunk_id}: {text[:60]}...")
        chunk_counter += 1

print(f"\nTotal amount of chunks: {len(chunks)}") 

Document: doc1_neural_networks.txt
  ├── chunk_01: An artificial neural network (ANN) is a computational model ...
  ├── chunk_02: The basic building block of a neural network is the perceptr...
  ├── chunk_03: Common activation functions include ReLU (Rectified Linear U...
Document: doc2_training.txt
  ├── chunk_04: Training a neural network involves adjusting its weights and...
  ├── chunk_05: Backpropagation is the core algorithm used to compute the gr...
  ├── chunk_06: Gradient descent and its variants like Adam update the model...
Document: doc3_architectures.txt
  ├── chunk_07: Convolutional Neural Networks (CNNs) are specialized for pro...
  ├── chunk_08: Recurrent Neural Networks (RNNs) are designed for sequential...
  ├── chunk_09: Long Short-Term Memory (LSTM) networks are an advanced type ...
Document: doc4_transformers.txt
  ├── chunk_10: The Transformer architecture relies entirely on the self-att...
  ├── chunk_11: Self-attention calculates attention scores between all p

In [3]:
texts_to_embed = [item["text"] for item in chunks]
embed_response = genai.embed_content(model=embed_model_name, content=texts_to_embed)
embeddings_list = embed_response["embedding"]

for item, vector in zip(chunks, embeddings_list):
    item["embedding"] = vector

sample = chunks[0]
print(f"Chunk ID: {sample['id']}")
print(f"Source: {sample['source']}")
print(f"Text: {sample['text']}")
print(f"Embedding dimension: {len(sample['embedding'])}")
print(f"Embedding vector preview: {[round(x, 4) for x in sample['embedding'][:8]]} ...")

Chunk ID: chunk_01
Source: doc1_neural_networks.txt
Text: An artificial neural network (ANN) is a computational model inspired by the structure of biological neural networks in the human brain. It consists of interconnected nodes called neurons organized into layers.
Embedding dimension: 3072
Embedding vector preview: [-0.0032, 0.0157, -0.0109, -0.0447, -0.0196, -0.0085, 0.0061, 0.0019] ...


In [4]:
class SimpleVectorStore:
    def __init__(self):
        self.records = []

    def add_items(self, items):
        for item in items:
            self.records.append({
                "id": item["id"],
                "text": item["text"],
                "embedding": item["embedding"],
                "source": item["source"]
            })

    def _cosine_similarity(self, vec_a, vec_b):
        dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
        norm_a = math.sqrt(sum(a * a for a in vec_a))
        norm_b = math.sqrt(sum(b * b for b in vec_b))
        return dot_product / (norm_a * norm_b) if norm_a and norm_b else 0.0

    def search(self, query_embedding, top_k=2):
        scored_results = []
        for record in self.records:
            score = self._cosine_similarity(query_embedding, record["embedding"])
            scored_results.append({
                "id": record["id"],
                "text": record["text"],
                "source": record["source"],
                "score": score
            })
        scored_results.sort(key=lambda x: x["score"], reverse=True)
        return scored_results[:top_k]

vector_store = SimpleVectorStore()
vector_store.add_items(chunks)

print(f"Vector Store initialized successfully. Stored vectors: {len(vector_store.records)}")

Vector Store initialized successfully. Stored vectors: 12


In [5]:
test_queries = [
    "What is a perceptron in neural networks?",
    "How does backpropagation calculate gradients?",
    "Which neural network architecture is best for images?",
    "How do LSTM networks solve the vanishing gradient problem?",
    "How does the self-attention mechanism work in Transformers?"
]

query_embeddings = genai.embed_content(model=embed_model_name, content=test_queries)["embedding"]

for i, (query, q_emb) in enumerate(zip(test_queries, query_embeddings), 1):
    results = vector_store.search(q_emb, top_k=2)
    print(f"Query {i}: {query}")
    for res in results:
        print(f"  -> [{res['score']:.4f}] ({res['source']} | {res['id']}): {res['text']}")
    print()

Query 1: What is a perceptron in neural networks?
  -> [0.7940] (doc1_neural_networks.txt | chunk_02): The basic building block of a neural network is the perceptron. A perceptron takes multiple numerical inputs, multiplies each by a weight, adds a bias, and passes the result through an activation function.
  -> [0.6530] (doc1_neural_networks.txt | chunk_01): An artificial neural network (ANN) is a computational model inspired by the structure of biological neural networks in the human brain. It consists of interconnected nodes called neurons organized into layers.

Query 2: How does backpropagation calculate gradients?
  -> [0.7943] (doc2_training.txt | chunk_05): Backpropagation is the core algorithm used to compute the gradient of the loss function with respect to each weight using the chain rule of calculus.
  -> [0.6707] (doc2_training.txt | chunk_06): Gradient descent and its variants like Adam update the model weights in the opposite direction of the gradient using a hyperparame

In [6]:
def run_rag(question, query_embedding=None, top_k=2):
    if query_embedding is None:
        query_embedding = genai.embed_content(model=embed_model_name, content=question)["embedding"]

    retrieved = vector_store.search(query_embedding, top_k=top_k)
    context_text = "\n".join([f"- {r['text']}" for r in retrieved])
    sources = ", ".join(sorted(set(r["source"] for r in retrieved)))

    prompt = f"""Using the provided context, answer the user's question concisely.
Context:
{context_text}

Question:
{question}

If the answer to the question is not in the context, explicitly state that the information is not available in the provided documents."""

    answer = gen_model.generate_content(prompt).text.strip()
    return {
        "question": question,
        "retrieved_text": retrieved[0]["text"],
        "sources": sources,
        "answer": answer
    }

rag_questions = [
    "What role do non-linear activation functions play in neural networks?",
    "What optimizer is mentioned as a variant of gradient descent?",
    "Why are CNNs used for image processing?",
    "What is the advantage of self-attention over recurrence?",
    "What accuracy and context window did NeuraLite-4 achieve?"
]

q_embeds = genai.embed_content(model=embed_model_name, content=rag_questions)["embedding"]
rag_results = []

for idx, (q, emb) in enumerate(zip(rag_questions, q_embeds), 1):
    if idx > 1:
        time.sleep(5)
    res = run_rag(q, query_embedding=emb, top_k=2)
    rag_results.append(res)
    print(f"Question {idx}")
    print(f"Query: {res['question']}")
    print(f"Top Fragment: {res['retrieved_text']}")
    print(f"Source: {res['sources']}")
    print(f"LLM Answer: {res['answer']}\n")

print("-" * 110)
print(f"{'No':<4} | {'Query':<32} | {'Source':<25} | {'LLM Answer'}")
print("-" * 110)
for idx, r in enumerate(rag_results, 1):
    short_q = (r["question"][:29] + "...") if len(r["question"]) > 32 else r["question"]
    short_a = r["answer"].replace("\n", " ")
    print(f"{idx:<4} | {short_q:<32} | {r['sources']:<25} | {short_a}")

Question 1
Query: What role do non-linear activation functions play in neural networks?
Top Fragment: Common activation functions include ReLU (Rectified Linear Unit), Sigmoid, and Tanh. Non-linear activation functions allow neural networks to learn complex patterns in data.
Source: doc1_neural_networks.txt
LLM Answer: Based on the provided context, non-linear activation functions allow neural networks to learn complex patterns in data.

Question 2
Query: What optimizer is mentioned as a variant of gradient descent?
Top Fragment: Gradient descent and its variants like Adam update the model weights in the opposite direction of the gradient using a hyperparameter called the learning rate.
Source: doc2_training.txt
LLM Answer: Adam

Question 3
Query: Why are CNNs used for image processing?
Top Fragment: Convolutional Neural Networks (CNNs) are specialized for processing grid-like data such as images. They use convolutional filters to automatically detect spatial hierarchies of features.
S

In [7]:
unknown_question = "What is the exact price and release date of the NVIDIA RTX 5090 graphics card?"

unknown_res = run_rag(unknown_question, top_k=2)

print(f"Question (Out of Context): {unknown_res['question']}")
print(f"Closest Retrieved Fragment: {unknown_res['retrieved_text']}")
print(f"Source Checked: {unknown_res['sources']}")
print(f"LLM Answer:\n{unknown_res['answer']}")

Question (Out of Context): What is the exact price and release date of the NVIDIA RTX 5090 graphics card?
Closest Retrieved Fragment: In our internal benchmark setup, the custom mini-transformer model NeuraLite-4 achieved 94.2% accuracy using 4 attention heads and a context window of 512 tokens.
Source Checked: doc1_neural_networks.txt, doc4_transformers.txt
LLM Answer:
The exact price and release date of the NVIDIA RTX 5090 graphics card are not available in the provided documents.


In [8]:
comparison_question = "What are the specifications and accuracy of the NeuraLite-4 model?"

answer_no_rag = gen_model.generate_content(comparison_question).text.strip()
time.sleep(5)
result_with_rag = run_rag(comparison_question, top_k=2)

print("1. LLM WITHOUT RAG")
print(answer_no_rag)

print("\n2. LLM WITH RAG")
print(f"Source used: {result_with_rag['sources']}")
print(result_with_rag["answer"])

1. LLM WITHOUT RAG
I do not have information on a model called "NeuraLite-4." It is possible this is a very new, proprietary, highly niche, or fictional AI model, or perhaps a typo for another system (such as Meta's Llama models, Google's Gemma, or similar lightweight architectures). 

If you meant a different model, please check the name and let me know, and I would be happy to provide its specifications and accuracy metrics.

2. LLM WITH RAG
Source used: doc1_neural_networks.txt, doc4_transformers.txt
Based on the provided context, the NeuraLite-4 model achieved 94.2% accuracy using 4 attention heads and a context window of 512 tokens.
